<a href="https://colab.research.google.com/github/Pazidu/Research-Project/blob/main/mobileNetV3Small.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/drive')

Mounted at /drive


In [2]:
import os
import shutil
import tensorflow as tf

from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    ReduceLROnPlateau,
    EarlyStopping
)

from tensorflow.keras import mixed_precision
from tensorflow.keras.applications import MobileNetV3Small
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input

print("TensorFlow version:", tf.__version__)
print("GPU:", tf.test.gpu_device_name())

TensorFlow version: 2.20.0
GPU: /device:GPU:0


In [3]:
BASE = "/content/newdata"

IMG_SRC = "/drive/MyDrive/Colab Notebooks/newdata"

CHECKPOINT_DIR = "/drive/MyDrive/checkpoints"

MODEL_SAVE_PATH = "/drive/MyDrive/Colab Notebooks/Models/dermoscopy/mobilenetv4_middle_fusion.keras"

In [4]:
if os.path.exists(BASE):
    shutil.rmtree(BASE)

shutil.copytree(IMG_SRC, BASE)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [5]:
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)

In [6]:
batch_size = 16
image_size = 224

In [7]:
data_augmentation = tf.keras.Sequential([

    layers.RandomFlip("horizontal"),

    layers.RandomRotation(0.1),

])

In [8]:
def add_edge_map(image, label):

    image = tf.cast(image, tf.float32)

    image = preprocess_input(image)

    gray = tf.image.rgb_to_grayscale(image)

    sobel = tf.image.sobel_edges(gray)

    edge = tf.sqrt(tf.reduce_sum(tf.square(sobel), axis=-1))

    edge = edge / (tf.reduce_max(edge) + 1e-6)

    edge = tf.cast(edge, tf.float32)

    return (image, edge), label

# =========================================================
# DATASET PREPARATION
# =========================================================

def prepare_dataset(path, shuffle):

    ds = tf.keras.preprocessing.image_dataset_from_directory(

        path,

        image_size=(image_size, image_size),

        batch_size=batch_size,

        label_mode='categorical',

        shuffle=shuffle

    )

    ds = ds.map(add_edge_map, num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds

train_ds = prepare_dataset(f"{BASE}/train", True)

val_ds = prepare_dataset(f"{BASE}/valid", False)

test_ds = prepare_dataset(f"{BASE}/test", False)

Found 8012 files belonging to 2 classes.
Found 1001 files belonging to 2 classes.
Found 1002 files belonging to 2 classes.


In [9]:
def create_dual_model():

    # -----------------------------------------------------
    # RGB INPUT
    # -----------------------------------------------------

    rgb_input = layers.Input(

        shape=(image_size, image_size, 3),

        name="rgb_input"

    )

    x_rgb = data_augmentation(rgb_input)

    # -----------------------------------------------------
    # BACKBONE
    # -----------------------------------------------------

    base = MobileNetV3Small(

        include_top=False,

        weights='imagenet',

        input_tensor=x_rgb

    )

    # Freeze most layers first
    base.trainable = True

    for layer in base.layers[:-30]:

        layer.trainable = False

    # -----------------------------------------------------
    # MIDDLE FEATURE MAP
    # -----------------------------------------------------

    middle_layer = base.get_layer("expanded_conv_6_project").output

    # -----------------------------------------------------
    # EDGE INPUT
    # -----------------------------------------------------

    edge_input = layers.Input(

        shape=(image_size, image_size, 1),

        name="edge_input"

    )

    x = layers.Conv2D(

        32,

        3,

        padding='same',

        activation='relu',

        dtype='float32'

    )(edge_input)

    x = layers.BatchNormalization(dtype='float32')(x)

    x = layers.MaxPooling2D(2)(x)

    x = layers.Dropout(0.2)(x)

    x = layers.Conv2D(

        64,

        3,

        padding='same',

        activation='relu',

        dtype='float32'

    )(x)

    x = layers.BatchNormalization(dtype='float32')(x)

    x = layers.MaxPooling2D(2)(x)

    x = layers.Dropout(0.2)(x)

    x = layers.Conv2D(

        128,

        3,

        padding='same',

        activation='relu',

        dtype='float32'

    )(x)

    x = layers.BatchNormalization(dtype='float32')(x)

    x = layers.MaxPooling2D(2)(x)

    x = layers.Dropout(0.3)(x)

    # -----------------------------------------------------
    # RESIZE EDGE FEATURES
    # -----------------------------------------------------

    target_size = middle_layer.shape[1:3]

    x = layers.Resizing(

        target_size[0],

        target_size[1]

    )(x)

    # -----------------------------------------------------
    # FEATURE FUSION
    # -----------------------------------------------------

    fused = layers.Concatenate()([middle_layer, x])

    # -----------------------------------------------------
    # POST-FUSION CNN
    # -----------------------------------------------------

    fused = layers.Conv2D(

        256,

        3,

        padding='same',

        activation='relu',

        dtype='float32'

    )(fused)

    fused = layers.BatchNormalization(dtype='float32')(fused)

    fused = layers.MaxPooling2D(2)(fused)

    fused = layers.Dropout(0.3)(fused)

    fused = layers.Conv2D(

        512,

        3,

        padding='same',

        activation='relu',

        dtype='float32'

    )(fused)

    fused = layers.BatchNormalization(dtype='float32')(fused)

    fused = layers.GlobalAveragePooling2D()(fused)

    fused = layers.Dropout(0.5)(fused)

    # -----------------------------------------------------
    # OUTPUT
    # -----------------------------------------------------

    outputs = layers.Dense(

        2,

        activation='softmax',

        dtype='float32'

    )(fused)

    model = Model(

        inputs=[rgb_input, edge_input],

        outputs=outputs

    )

    # -----------------------------------------------------
    # COMPILE
    # -----------------------------------------------------

    model.compile(

        optimizer=tf.keras.optimizers.Adam(1e-4),

        loss=tf.keras.losses.CategoricalCrossentropy(
            label_smoothing=0.05
        ),

        metrics=['accuracy']

    )

    return model

In [10]:
model = create_dual_model()

model.summary()

4334752/4334752 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ rgb_input           │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential          │ (None, 224, 224,  │          0 │ rgb_input[0][0]   │
│ (Sequential)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 224, 224,  │          0 │ sequential[0][0]  │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv (Conv2D)       │ (None, 112, 112,  │        432 │ rescaling[0][0]   │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_bn             │ (None, 112, 112,  │         64 │ conv[0][0]        │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 112, 112,  │          0 │ conv_bn[0][0]     │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 113, 113,  │          0 │ activation[0][0]  │
│ (ZeroPadding2D)     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 56, 56,    │        144 │ expanded_conv_de… │
│ (DepthwiseConv2D)   │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 56, 56,    │         64 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu (ReLU)        │ (None, 56, 56,    │          0 │ expanded_conv_de… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_sque… │ (None, 1, 1, 16)  │          0 │ re_lu[0][0]       │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_sque… │ (None, 1, 1, 8)   │        136 │ expanded_conv_sq… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_sque… │ (None, 1, 1, 8)   │          0 │ expanded_conv_sq… │
│ (ReLU)              │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_sque… │ (None, 1, 1, 16)  │        144 │ expanded_conv_sq… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 1, 1, 16)  │          0 │ expanded_conv_sq… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_1 (ReLU)      │ (None, 1, 1, 16)  │          0 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 1, 1, 16)  │          0 │ re_lu_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_sque… │ (None, 56, 56,    │          0 │ re_lu[0][0],    

 Total params: 1,848,066 (7.05 MB)

 Trainable params: 1,681,602 (6.41 MB)

 Non-trainable params: 166,464 (650.25 KB)

In [11]:
checkpoint_best = ModelCheckpoint(

    filepath=f"{CHECKPOINT_DIR}/best_model.keras",

    monitor="val_accuracy",

    save_best_only=True,

    verbose=1

)

reduce_lr = ReduceLROnPlateau(

    monitor='val_loss',

    factor=0.5,

    patience=3,

    verbose=1,

    min_lr=1e-7

)

early_stop = EarlyStopping(

    monitor='val_loss',

    patience=5,

    restore_best_weights=True,

    verbose=1

)

In [12]:
print("\n==============================")
print("TRAINING")
print("==============================\n")

history = model.fit(

    train_ds,

    epochs=15,

    validation_data=val_ds,

    callbacks=[

        checkpoint_best,

        reduce_lr,

        early_stop

    ]

)


TRAINING

Epoch 1/15
501/501 ━━━━━━━━━━━━━━━━━━━━ 0s 232ms/step - accuracy: 0.6939 - loss: 0.6108
Epoch 1: val_accuracy improved from None to 0.83117, saving model to /drive/MyDrive/checkpoints/best_model.keras

Epoch 1: finished saving model to /drive/MyDrive/checkpoints/best_model.keras
501/501 ━━━━━━━━━━━━━━━━━━━━ 165s 284ms/step - accuracy: 0.7776 - loss: 0.5250 - val_accuracy: 0.8312 - val_loss: 0.4352 - learning_rate: 1.0000e-04
Epoch 2/15
501/501 ━━━━━━━━━━━━━━━━━━━━ 0s 205ms/step - accuracy: 0.8708 - loss: 0.3757
Epoch 2: val_accuracy improved from 0.83117 to 0.87912, saving model to /drive/MyDrive/checkpoints/best_model.keras

Epoch 2: finished saving model to /drive/MyDrive/checkpoints/best_model.keras
501/501 ━━━━━━━━━━━━━━━━━━━━ 116s 231ms/step - accuracy: 0.8778 - loss: 0.3639 - val_accuracy: 0.8791 - val_loss: 0.3592 - learning_rate: 1.0000e-04
Epoch 3/15
501/501 ━━━━━━━━━━━━━━━━━━━━ 0s 202ms/step - accuracy: 0.8848 - loss: 0.3518
Epoch 3: val_accuracy did not improve fr

In [13]:
print("\n==============================")
print("TEST EVALUATION")
print("==============================\n")

loss, acc = model.evaluate(test_ds)

print(f"\nFinal Test Accuracy: {acc:.4f}")


TEST EVALUATION

63/63 ━━━━━━━━━━━━━━━━━━━━ 20s 314ms/step - accuracy: 0.8493 - loss: 0.3793

Final Test Accuracy: 0.8493


In [14]:
model.save(MODEL_SAVE_PATH)

print("\nModel Saved Successfully!")


Model Saved Successfully!
